In [0]:
# Databricks notebook
# Purpose: Create and load the Bronze reference Delta table ref_de_para_ramo_susep_raw
# Source: proposta_ramos_tratados_susep.xlsx / sheet 01_proposta_ramos

from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, DateType, StringType, StructField, StructType, TimestampType

#  ============================================================
# 1. CONFIGURATION
#  ============================================================

database_name = "susep_bronze"
table_name = "aux_de_para_ramo_susep"
full_table_name = f"{database_name}.{table_name}"


# Regra de vigencia da proposta.
# Use 1900-01-01 para aplicar o de/para retroativamente em series historicas.
vigencia_inicio_default = "1900-01-01"
vigencia_fim_default = None
flag_ativo_default = True
versao_mapeamento_default = "v1.0"
fonte_regra_default = "Proposta interna baseada em tabela de grupos e ramos SUSEP"
write_mode = "overwrite"

print(f"Target table: {full_table_name}")


In [0]:
# ============================================================
# 2. REFERENCE DATA
# ============================================================
# Estrutura base: cod_grupo_susep, nome_grupo_susep, cod_ramo_susep,
# nome_ramo_susep, cod_ramo_tratado, nome_ramo_tratado,
# justificativa_tratamento.

mapping_data = [('01',
  'Patrimonial',
  '0111',
  'INCÊNDIO TRADICIONAL(RUN OFF)',
  '0101',
  'Incêndio e Compreensivos Patrimoniais',
  'Família patrimonial de incêndio/compreensivos; ramo 0111 é legado/run-off.'),
 ('01',
  'Patrimonial',
  '0112',
  'Assistência - Bens em Geral',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Coberturas amplas ou acessórias patrimoniais, sem ramo mais específico no tratamento.'),
 ('01',
  'Patrimonial',
  '0113',
  'Vidros',
  '0103',
  'Vidros',
  'Ramo específico mantido pela materialidade técnica própria.'),
 ('01',
  'Patrimonial',
  '0114',
  'Compreensivo Residencial',
  '0101',
  'Incêndio e Compreensivos Patrimoniais',
  'Família patrimonial de incêndio/compreensivos; ramo 0111 é legado/run-off.'),
 ('01',
  'Patrimonial',
  '0115',
  'ROUBO(RUN OFF)',
  '0104',
  'Roubo e Tumultos',
  'Coberturas patrimoniais de eventos específicos, incluindo ramo legado/run-off.'),
 ('01',
  'Patrimonial',
  '0116',
  'Compreensivo Condomínio',
  '0101',
  'Incêndio e Compreensivos Patrimoniais',
  'Família patrimonial de incêndio/compreensivos; ramo 0111 é legado/run-off.'),
 ('01',
  'Patrimonial',
  '0117',
  'Tumultos',
  '0104',
  'Roubo e Tumultos',
  'Coberturas patrimoniais de eventos específicos, incluindo ramo legado/run-off.'),
 ('01',
  'Patrimonial',
  '0118',
  'Compreensivo Empresarial',
  '0101',
  'Incêndio e Compreensivos Patrimoniais',
  'Família patrimonial de incêndio/compreensivos; ramo 0111 é legado/run-off.'),
 ('01',
  'Patrimonial',
  '0141',
  'Lucros Cessantes',
  '0105',
  'Lucros Cessantes',
  'Mesmo objeto econômico: perda de resultado/continuidade operacional.'),
 ('01',
  'Patrimonial',
  '0142',
  'Lucros Cessantes Cobertura Simples',
  '0105',
  'Lucros Cessantes',
  'Mesmo objeto econômico: perda de resultado/continuidade operacional.'),
 ('01',
  'Patrimonial',
  '0143',
  'Fidelidade',
  '0106',
  'Fidelidade e Global de Bancos',
  'Riscos de fidelidade/crime financeiro em bens/valores.'),
 ('01',
  'Patrimonial',
  '0167',
  'Riscos de Engenharia',
  '0107',
  'Riscos de Engenharia, Nomeados e Operacionais',
  'Riscos patrimoniais empresariais/operacionais de maior complexidade.'),
 ('01',
  'Patrimonial',
  '0171',
  'Riscos Diversos',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Coberturas amplas ou acessórias patrimoniais, sem ramo mais específico no tratamento.'),
 ('01',
  'Patrimonial',
  '0173',
  'Global de Bancos',
  '0106',
  'Fidelidade e Global de Bancos',
  'Riscos de fidelidade/crime financeiro em bens/valores.'),
 ('01',
  'Patrimonial',
  '0176',
  'Riscos Diversos - Planos Conjugados',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Coberturas amplas ou acessórias patrimoniais, sem ramo mais específico no tratamento.'),
 ('01',
  'Patrimonial',
  '0195',
  'Garantia Est./Ext.Gar-Bens em Geral',
  '0108',
  'Garantia Estendida - Bens em Geral',
  'Ramo de garantia estendida para bens em geral, separado de auto.'),
 ('01',
  'Patrimonial',
  '0196',
  'Riscos Nomeados e Operacionais',
  '0107',
  'Riscos de Engenharia, Nomeados e Operacionais',
  'Riscos patrimoniais empresariais/operacionais de maior complexidade.'),
 ('02',
  'Riscos Especiais',
  '0234',
  'RISCOS DE PETRÓLEO(RUN OFF)',
  '0201',
  'Riscos de Petróleo - Run Off',
  'Ramo legado em grupo de riscos especiais; mantido separado por natureza técnica.'),
 ('02',
  'Riscos Especiais',
  '0272',
  'RISCOS NUCLEARES(RUN OFF)',
  '0202',
  'Riscos Nucleares - Run Off',
  'Ramo legado em grupo de riscos especiais; mantido separado por natureza técnica.'),
 ('02',
  'Riscos Especiais',
  '0274',
  'SATÉLITES(RUN OFF)',
  '0203',
  'Satélites - Run Off',
  'Ramo legado em grupo de riscos especiais; mantido separado por natureza técnica.'),
 ('03',
  'Responsabilidades',
  '0310',
  'R.C.Administradores e Diretores-D&O',
  '0301',
  'Responsabilidade Civil Profissional e D&O',
  'Linhas de RC profissional/executivos com lógica de gestão de responsabilidade especializada.'),
 ('03',
  'Responsabilidades',
  '0313',
  'R. C. Riscos Ambientais',
  '0302',
  'Responsabilidade Civil Ambiental',
  'Risco de responsabilidade ambiental tratado separadamente.'),
 ('03',
  'Responsabilidades',
  '0327',
  'Compreensivo Riscos Cibernéticos',
  '0303',
  'Riscos Cibernéticos',
  'Linha cyber mantida separada por dinâmica de risco e produto.'),
 ('03',
  'Responsabilidades',
  '0351',
  'R. C. Geral',
  '0304',
  'Responsabilidade Civil Geral',
  'Responsabilidade civil ampla, sem especialização profissional/ambiental/cyber.'),
 ('03',
  'Responsabilidades',
  '0378',
  'R. C. Profissional',
  '0301',
  'Responsabilidade Civil Profissional e D&O',
  'Linhas de RC profissional/executivos com lógica de gestão de responsabilidade especializada.'),
 ('04',
  'Cascos',
  '0433',
  'Marítimos(RUN OFF)',
  '0401',
  'Cascos Marítimos - Run Off',
  'Ramo legado de cascos marítimos dentro do grupo Cascos.'),
 ('04',
  'Cascos',
  '0435',
  'AERONÁUTICOS(RUN OFF)',
  '0402',
  'Cascos Aeronáuticos',
  'Cascos aeronáuticos, incluindo bilhete e legado/run-off.'),
 ('04',
  'Cascos',
  '0437',
  'RESPONSAB CIVIL HANGAR(RUN OFF)',
  '0403',
  'Responsabilidade Civil Hangar - Run Off',
  'Ramo legado de RC Hangar mantido separado dentro do grupo Cascos.'),
 ('04',
  'Cascos',
  '0457',
  'D. P. E. M.(RUN OFF)',
  '0404',
  'DPEM - Run Off',
  'Ramo obrigatório marítimo em run-off mantido separado.'),
 ('04',
  'Cascos',
  '0484',
  'Aeronáuticos - Bilhete',
  '0402',
  'Cascos Aeronáuticos',
  'Cascos aeronáuticos, incluindo bilhete e legado/run-off.'),
 ('05',
  'Automóvel',
  '0520',
  'Acidentes Pessoais Passageiros-APP',
  '0501',
  'APP - Acidentes Pessoais de Passageiros',
  'Acidentes pessoais de passageiros vinculados ao automóvel.'),
 ('05',
  'Automóvel',
  '0523',
  'RC T.ROD.PAS INTEST OU INT(RUN OFF)',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'RC vinculada a veículos/acordos de circulação; sem cruzar com grupos de Transportes.'),
 ('05',
  'Automóvel',
  '0524',
  'Garantia Est./ Exten. Garantia – Auto',
  '0503',
  'Garantia Estendida - Auto',
  'Garantia estendida automotiva mantida separada de garantia de bens.'),
 ('05',
  'Automóvel',
  '0525',
  'Carta Verde',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'RC vinculada a veículos/acordos de circulação; sem cruzar com grupos de Transportes.'),
 ('05',
  'Automóvel',
  '0526',
  'Seguro Popular de Auto Us (RUN OFF)',
  '0504',
  'Automóvel - Casco e Auto Popular',
  'Cobertura principal do veículo; Auto Popular é legado/run-off da família casco.'),
 ('05',
  'Automóvel',
  '0527',
  'RC Veic Pass Acordos fora Mercosul',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'RC vinculada a veículos/acordos de circulação; sem cruzar com grupos de Transportes.'),
 ('05',
  'Automóvel',
  '0531',
  'Automóvel - Casco',
  '0504',
  'Automóvel - Casco e Auto Popular',
  'Cobertura principal do veículo; Auto Popular é legado/run-off da família casco.'),
 ('05',
  'Automóvel',
  '0542',
  'Assistência e Outras Cobert. - Auto',
  '0505',
  'Assistência e Outras Coberturas - Auto',
  'Coberturas acessórias do veículo.'),
 ('05',
  'Automóvel',
  '0544',
  'R.C.T.Viag InterPes Tran/ñ(RUN OFF)',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'RC vinculada a veículos/acordos de circulação; sem cruzar com grupos de Transportes.'),
 ('05',
  'Automóvel',
  '0553',
  'R. C. Facultativa Veículos - RCFV',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'RC vinculada a veículos/acordos de circulação; sem cruzar com grupos de Transportes.'),
 ('05', 'Automóvel', '0588', 'DPVAT', '0506', 'DPVAT', 'DPVAT ativo/legado tratado como família própria.'),
 ('05', 'Automóvel', '0589', 'DPVAT(RUN OFF)', '0506', 'DPVAT', 'DPVAT ativo/legado tratado como família própria.'),
 ('06',
  'Transportes',
  '0621',
  'Transporte Nacional',
  '0601',
  'Transporte Nacional e Internacional',
  'Transporte de mercadorias/cargas por origem/destino.'),
 ('06',
  'Transportes',
  '0622',
  'Transporte Internacional',
  '0601',
  'Transporte Nacional e Internacional',
  'Transporte de mercadorias/cargas por origem/destino.'),
 ('06',
  'Transportes',
  '0623',
  'RCTR-P Interestadual/Internacional',
  '0602',
  'RC Transporte de Passageiros e Viagem Internacional',
  'RC do transportador de passageiros e viagens internacionais.'),
 ('06',
  'Transportes',
  '0627',
  'Resp. Civil do Transp.Intm(RUN OFF)',
  '0603',
  'RC Operador Multimodal/Intermodal',
  'Ramos de responsabilidade do operador inter/multimodal.'),
 ('06',
  'Transportes',
  '0628',
  'RCTR-P Municipal/Intermunicipal',
  '0602',
  'RC Transporte de Passageiros e Viagem Internacional',
  'RC do transportador de passageiros e viagens internacionais.'),
 ('06',
  'Transportes',
  '0632',
  'R.C.Trans.Carga Viag.Int.-RCTR-VI-C',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0638',
  'R.C.Trans. Ferroviário Carga – RCTF-C',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0644',
  'R.C. Viag.Int. Pessoas - Carta Azul',
  '0602',
  'RC Transporte de Passageiros e Viagem Internacional',
  'RC do transportador de passageiros e viagens internacionais.'),
 ('06',
  'Transportes',
  '0645',
  'RC Transp Intern fora do ATIT',
  '0602',
  'RC Transporte de Passageiros e Viagem Internacional',
  'RC do transportador de passageiros e viagens internacionais.'),
 ('06',
  'Transportes',
  '0652',
  'R. C. Trans. Aéreo Carga - RCTA-C',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0654',
  'R.C. Trans. Rodoviário Carga – RCTR-C',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0655',
  'RC Trans Desaparec Carga RC-DC',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0656',
  'R.C. Trans. Aquaviário Carga – RCA-C',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('06',
  'Transportes',
  '0658',
  'R.C.Operador Transp. Multi.-RCOTM-C',
  '0603',
  'RC Operador Multimodal/Intermodal',
  'Ramos de responsabilidade do operador inter/multimodal.'),
 ('06',
  'Transportes',
  '0659',
  'RC Veículo Transp Rod Carga RC-V',
  '0604',
  'RC Transporte de Carga',
  'RC do transportador/veículo de carga por modal e coberturas correlatas.'),
 ('07',
  'Riscos Financeiros',
  '0711',
  'Riscos Diversos Financeiros',
  '0701',
  'Riscos Diversos Financeiros',
  'Ramo financeiro residual/específico.'),
 ('07',
  'Riscos Financeiros',
  '0739',
  'Garantia Financeira(RUN OFF)',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07',
  'Riscos Financeiros',
  '0740',
  'Garantia de Obrigaçõe Priv(RUN OFF)',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07', 'Riscos Financeiros', '0743', 'Stop Loss', '0703', 'Stop Loss', 'Ramo específico de stop loss.'),
 ('07',
  'Riscos Financeiros',
  '0745',
  'Garantia de Obrig Públicas(RUN OFF)',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07',
  'Riscos Financeiros',
  '0746',
  'Fiança Locatícia',
  '0704',
  'Fiança Locatícia',
  'Ramo específico de fiança locatícia.'),
 ('07',
  'Riscos Financeiros',
  '0747',
  'Garantia de Conc Públicas (RUN OFF)',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07',
  'Riscos Financeiros',
  '0748',
  'Crédito Interno',
  '0705',
  'Crédito - Riscos Financeiros',
  'Crédito interno/exportação dentro do grupo 07, sem juntar com grupo 08.'),
 ('07',
  'Riscos Financeiros',
  '0749',
  'Crédito à Exportação',
  '0705',
  'Crédito - Riscos Financeiros',
  'Crédito interno/exportação dentro do grupo 07, sem juntar com grupo 08.'),
 ('07',
  'Riscos Financeiros',
  '0750',
  'Garantia Judicial(RUN OFF)',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07',
  'Riscos Financeiros',
  '0775',
  'Garantia Segurado - Setor Público',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('07',
  'Riscos Financeiros',
  '0776',
  'Garantia Segurado - Setor Privado',
  '0702',
  'Garantia',
  'Ramos de seguro garantia: legado/run-off e atuais, separados de Crédito.'),
 ('08',
  'Crédito',
  '0819',
  'Crédito à Exp. Risco Comer(RUN OFF)',
  '0801',
  'Crédito à Exportação',
  'Crédito à exportação, agregando risco comercial/político legados.'),
 ('08',
  'Crédito',
  '0848',
  'Crédito Interno',
  '0802',
  'Crédito Interno/Doméstico',
  'Crédito interno e antigos ramos domésticos em run-off.'),
 ('08',
  'Crédito',
  '0849',
  'Crédito à Exportação',
  '0801',
  'Crédito à Exportação',
  'Crédito à exportação, agregando risco comercial/político legados.'),
 ('08',
  'Crédito',
  '0859',
  'Crédito à Expor Risc Polít(RUN OFF)',
  '0801',
  'Crédito à Exportação',
  'Crédito à exportação, agregando risco comercial/político legados.'),
 ('08',
  'Crédito',
  '0860',
  'Crédito Dom Risco Comerc(RUN OFF)',
  '0802',
  'Crédito Interno/Doméstico',
  'Crédito interno e antigos ramos domésticos em run-off.'),
 ('08',
  'Crédito',
  '0870',
  'Crédito Dom Risco P.Físic (RUN OFF)',
  '0802',
  'Crédito Interno/Doméstico',
  'Crédito interno e antigos ramos domésticos em run-off.'),
 ('09', 'Pessoas Coletivo', '0929', 'Funeral', '0901', 'Funeral', 'Ramo de pessoas coletivo com objeto específico.'),
 ('09',
  'Pessoas Coletivo',
  '0936',
  'Perda Certif. Habilit. de Vôo-PCHV',
  '0902',
  'PCHV',
  'Perda de Certificado de Habilitação de Voo no grupo coletivo.'),
 ('09', 'Pessoas Coletivo', '0969', 'Viagem', '0903', 'Viagem', 'Seguro viagem coletivo.'),
 ('09',
  'Pessoas Coletivo',
  '0977',
  'Prestamista (exceto Habit e Rural)',
  '0904',
  'Prestamista',
  'Prestamista coletivo, exceto habitacional e rural.'),
 ('09', 'Pessoas Coletivo', '0980', 'Educacional', '0905', 'Educacional', 'Seguro educacional coletivo.'),
 ('09',
  'Pessoas Coletivo',
  '0981',
  'ACIDENTES PESS - INDIVID(RUN OFF)',
  '0906',
  'Acidentes Pessoais',
  'Acidentes pessoais, incluindo legado/run-off.'),
 ('09',
  'Pessoas Coletivo',
  '0982',
  'Acidentes Pessoais',
  '0906',
  'Acidentes Pessoais',
  'Acidentes pessoais, incluindo legado/run-off.'),
 ('09', 'Pessoas Coletivo', '0983', 'Dotal Misto', '0907', 'Dotais', 'Dotal misto e puro por afinidade de produto.'),
 ('09',
  'Pessoas Coletivo',
  '0984',
  'Doenças Graves ou Doença Terminal',
  '0908',
  'Doenças Graves ou Doença Terminal',
  'Linha de pessoas de risco biométrico específica.'),
 ('09', 'Pessoas Coletivo', '0986', 'Dotal Puro', '0907', 'Dotais', 'Dotal misto e puro por afinidade de produto.'),
 ('09',
  'Pessoas Coletivo',
  '0987',
  'Desemprego/Perda de Renda',
  '0909',
  'Desemprego/Perda de Renda',
  'Renda/proteção financeira de pessoas coletivo.'),
 ('09',
  'Pessoas Coletivo',
  '0990',
  'Eventos Aleatórios',
  '0910',
  'Eventos Aleatórios',
  'Ramo residual para riscos de pessoas sem ramo próprio.'),
 ('09',
  'Pessoas Coletivo',
  '0991',
  'Vida(RUN OFF)',
  '0911',
  'Vida e VG/APC',
  'Vida coletiva, incluindo Vida em Grupo e VG/APC legados/específicos.'),
 ('09',
  'Pessoas Coletivo',
  '0992',
  'VGBL/VAGP/VRGP/VRSA/PRI ind(RUN OFF',
  '0912',
  'Sobrevivência/VGBL-VAGP-VRGP-VRSA-VRID-VDR',
  'Produtos/coberturas por sobrevivência e previdenciários no grupo coletivo.'),
 ('09',
  'Pessoas Coletivo',
  '0993',
  'Vida em Grupo',
  '0911',
  'Vida e VG/APC',
  'Vida coletiva, incluindo Vida em Grupo e VG/APC legados/específicos.'),
 ('09',
  'Pessoas Coletivo',
  '0994',
  'VGBL/ VAGP/ VRGP/ VRSA/ VRID/ VDR',
  '0912',
  'Sobrevivência/VGBL-VAGP-VRGP-VRSA-VRID-VDR',
  'Produtos/coberturas por sobrevivência e previdenciários no grupo coletivo.'),
 ('09',
  'Pessoas Coletivo',
  '0997',
  'VG/APC',
  '0911',
  'Vida e VG/APC',
  'Vida coletiva, incluindo Vida em Grupo e VG/APC legados/específicos.'),
 ('10',
  'Habitacional',
  '1061',
  'Seg.Habit.Apól. Merc. - Prestamista',
  '1001',
  'Habitacional - Prestamista',
  'MIP/prestamista em apólices de mercado.'),
 ('10',
  'Habitacional',
  '1065',
  'Seg.Habit.Apól.Merc.-Demais Cobert.',
  '1002',
  'Habitacional - Demais Coberturas',
  'DFI e demais coberturas habitacionais em apólices de mercado.'),
 ('10',
  'Habitacional',
  '1066',
  'Seg.Hab.Sist.Fin. da Habit(RUN OFF)',
  '1003',
  'Habitacional - Run Off/SFH',
  'Ramos habitacionais legados/run-off, sem misturar com apólice de mercado ativa.'),
 ('10',
  'Habitacional',
  '1068',
  'HABITACIONAL - FORA DO SFH(RUN OFF)',
  '1003',
  'Habitacional - Run Off/SFH',
  'Ramos habitacionais legados/run-off, sem misturar com apólice de mercado ativa.'),
 ('11',
  'Rural',
  '1101',
  'Seguro Agr sem cob do FESR(RUN OFF)',
  '1101',
  'Agrícola',
  'Seguros agrícolas atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1102',
  'Seguro Agr com cob do FESR(RUN OFF)',
  '1101',
  'Agrícola',
  'Seguros agrícolas atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1103',
  'Seguro Pec sem cob do FESR(RUN OFF)',
  '1102',
  'Pecuário e Animais',
  'Pecuário/animais atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1104',
  'Seguro Pec com cob do FESR(RUN OFF)',
  '1102',
  'Pecuário e Animais',
  'Pecuário/animais atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1105',
  'Seguro Aq sem cob do FESR(RUN OFF)',
  '1103',
  'Aquícola',
  'Aquícola atual e legado com/sem FESR.'),
 ('11',
  'Rural',
  '1106',
  'Seguro Aq com cob do FESR(RUN OFF)',
  '1103',
  'Aquícola',
  'Aquícola atual e legado com/sem FESR.'),
 ('11',
  'Rural',
  '1107',
  'Seguro Flore s/cob do FESR(RUN OFF)',
  '1104',
  'Florestas',
  'Florestas atuais e legadas com/sem FESR.'),
 ('11',
  'Rural',
  '1108',
  'Seguro Flore c coB do FESR(RUN OFF)',
  '1104',
  'Florestas',
  'Florestas atuais e legadas com/sem FESR.'),
 ('11',
  'Rural',
  '1109',
  'Seguro da Céd do Pr Rural(RUN OFF)',
  '1105',
  'Cédula do Produto Rural - Run Off',
  'Ramo rural legado indicado como run-off.'),
 ('11', 'Rural', '1111', 'Seguro Agrícola', '1101', 'Agrícola', 'Seguros agrícolas atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1112',
  'Seguro Pecuário',
  '1102',
  'Pecuário e Animais',
  'Pecuário/animais atuais e legados com/sem FESR.'),
 ('11', 'Rural', '1113', 'Seguro Aquícola', '1103', 'Aquícola', 'Aquícola atual e legado com/sem FESR.'),
 ('11', 'Rural', '1114', 'Seguro Florestas', '1104', 'Florestas', 'Florestas atuais e legadas com/sem FESR.'),
 ('11', 'Rural', '1128', 'Pecuário', '1102', 'Pecuário e Animais', 'Pecuário/animais atuais e legados com/sem FESR.'),
 ('11', 'Rural', '1129', 'Aquícola', '1103', 'Aquícola', 'Aquícola atual e legado com/sem FESR.'),
 ('11',
  'Rural',
  '1130',
  'Seguro Benf. e Prod. Agropecuários',
  '1106',
  'Bens, Benfeitorias e Penhor Rural',
  'Bens agropecuários, benfeitorias e penhor rural.'),
 ('11', 'Rural', '1161', 'Agrícola', '1101', 'Agrícola', 'Seguros agrícolas atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1162',
  'Penhor Rural',
  '1106',
  'Bens, Benfeitorias e Penhor Rural',
  'Bens agropecuários, benfeitorias e penhor rural.'),
 ('11',
  'Rural',
  '1163',
  'Penhor Rural Inst. Fin Pub(RUN OFF)',
  '1106',
  'Bens, Benfeitorias e Penhor Rural',
  'Bens agropecuários, benfeitorias e penhor rural.'),
 ('11',
  'Rural',
  '1164',
  'Seguros Animais',
  '1102',
  'Pecuário e Animais',
  'Pecuário/animais atuais e legados com/sem FESR.'),
 ('11',
  'Rural',
  '1165',
  'Compreensivo de Florestas',
  '1104',
  'Florestas',
  'Florestas atuais e legadas com/sem FESR.'),
 ('11',
  'Rural',
  '1198',
  'Seguro de Vida do Produtor Rural',
  '1107',
  'Vida do Produtor Rural',
  'Seguro de vida do produtor rural devedor de crédito rural.'),
 ('12',
  'Outros',
  '1279',
  'Seguros no Exterior(RUN OFF)',
  '1201',
  'Exterior e Sucursais no Exterior',
  'Operações no exterior/sucursais em run-off.'),
 ('12',
  'Outros',
  '1285',
  'Saúde - Ressegurador Loc (RUN OFF)',
  '1202',
  'Saúde',
  'Saúde individual/grupal e ressegurador local legado dentro do grupo Outros.'),
 ('12',
  'Outros',
  '1286',
  'Saúde Individual',
  '1202',
  'Saúde',
  'Saúde individual/grupal e ressegurador local legado dentro do grupo Outros.'),
 ('12',
  'Outros',
  '1287',
  'Saúde Grupal',
  '1202',
  'Saúde',
  'Saúde individual/grupal e ressegurador local legado dentro do grupo Outros.'),
 ('12',
  'Outros',
  '1299',
  'SUCURSAIS NO EXTERIOR(RUN OFF)',
  '1201',
  'Exterior e Sucursais no Exterior',
  'Operações no exterior/sucursais em run-off.'),
 ('13',
  'Pessoas Individual',
  '1329',
  'Funeral',
  '1301',
  'Funeral',
  'Ramo de pessoas individual com objeto específico.'),
 ('13',
  'Pessoas Individual',
  '1336',
  'Perda Certif. Habilit. de Vôo-PCHV',
  '1302',
  'PCHV',
  'Perda de Certificado de Habilitação de Voo no grupo individual.'),
 ('13', 'Pessoas Individual', '1369', 'Viagem', '1303', 'Viagem', 'Seguro viagem individual.'),
 ('13',
  'Pessoas Individual',
  '1377',
  'Prestamista (exceto Habit. E Rural)',
  '1304',
  'Prestamista',
  'Prestamista individual, exceto habitacional e rural.'),
 ('13', 'Pessoas Individual', '1380', 'Educacional', '1305', 'Educacional', 'Seguro educacional individual.'),
 ('13',
  'Pessoas Individual',
  '1381',
  'Acidentes Pessoais',
  '1306',
  'Acidentes Pessoais',
  'Acidentes pessoais individual.'),
 ('13',
  'Pessoas Individual',
  '1383',
  'Dotal Misto',
  '1307',
  'Dotais',
  'Dotal misto e puro por afinidade de produto.'),
 ('13',
  'Pessoas Individual',
  '1384',
  'Doenças Graves ou Doença Terminal',
  '1308',
  'Doenças Graves ou Doença Terminal',
  'Linha de pessoas de risco biométrico específica.'),
 ('13', 'Pessoas Individual', '1386', 'Dotal Puro', '1307', 'Dotais', 'Dotal misto e puro por afinidade de produto.'),
 ('13',
  'Pessoas Individual',
  '1387',
  'Desemprego/Perda de Renda',
  '1309',
  'Desemprego/Perda de Renda',
  'Renda/proteção financeira de pessoas individual.'),
 ('13',
  'Pessoas Individual',
  '1390',
  'Eventos Aleatórios',
  '1310',
  'Eventos Aleatórios',
  'Ramo residual para riscos de pessoas sem ramo próprio.'),
 ('13', 'Pessoas Individual', '1391', 'Vida', '1311', 'Vida', 'Vida individual.'),
 ('13',
  'Pessoas Individual',
  '1392',
  'VGBL/ VAGP/ VRGP/ VRSA/ VRID/ VDR',
  '1312',
  'Sobrevivência/VGBL-VAGP-VRGP-VRSA-VRID-VDR',
  'Produtos/coberturas por sobrevivência e previdenciários no grupo individual.'),
 ('14',
  'Marítimos',
  '1417',
  'Seg. Compreensivo Oper. Portuários',
  '1401',
  'Operações Portuárias',
  'Seguro compreensivo de operações portuárias.'),
 ('14',
  'Marítimos',
  '1428',
  'R. C. Facult. para Embarcações-RCF',
  '1402',
  'Responsabilidade Civil para Embarcações',
  'RC facultativa para embarcações.'),
 ('14',
  'Marítimos',
  '1433',
  'Marítimos (Cascos)',
  '1403',
  'Cascos Marítimos',
  'Cascos marítimos no grupo Marítimos.'),
 ('14', 'Marítimos', '1457', 'DPEM', '1404', 'DPEM', 'Danos pessoais causados por embarcações ou carga.'),
 ('15',
  'Aeronáuticos',
  '1528',
  'R. C. Facult. para Aeronaves - RCF',
  '1501',
  'Responsabilidade Civil Aeronáutica',
  'RC aeronaves/hangar/explorador ou transportador aéreo.'),
 ('15', 'Aeronáuticos', '1535', 'Aeronáuticos (cascos)', '1502', 'Cascos Aeronáuticos', 'Cascos aeronáuticos.'),
 ('15',
  'Aeronáuticos',
  '1537',
  'Responsabilidade Civil Hangar',
  '1501',
  'Responsabilidade Civil Aeronáutica',
  'RC aeronaves/hangar/explorador ou transportador aéreo.'),
 ('15', 'Aeronáuticos', '1574', 'Satélites', '1503', 'Satélites', 'Satélites no grupo Aeronáuticos.'),
 ('15',
  'Aeronáuticos',
  '1597',
  'Resp. Explor. ou Transp. Aéreo-RETA',
  '1501',
  'Responsabilidade Civil Aeronáutica',
  'RC aeronaves/hangar/explorador ou transportador aéreo.'),
 ('16',
  'Microsseguros',
  '1601',
  'Microsseguros de Pessoas',
  '1601',
  'Microsseguros de Pessoas',
  'Microsseguro de pessoas.'),
 ('16',
  'Microsseguros',
  '1602',
  'Microsseguros de Danos',
  '1602',
  'Microsseguros de Danos',
  'Microsseguro de danos.'),
 ('16',
  'Microsseguros',
  '1603',
  'Microsseguros - Previdência',
  '1603',
  'Microsseguros - Previdência',
  'Microsseguro previdenciário.'),
 ('17', 'Petróleo', '1734', 'Riscos de Petróleo', '1701', 'Riscos de Petróleo', 'Grupo específico de petróleo.'),
 ('18', 'Nucleares', '1872', 'Riscos Nucleares', '1801', 'Riscos Nucleares', 'Grupo específico de nucleares.'),
 ('19', 'Saúde', '1985', 'Saúde – Resseguro', '1901', 'Saúde - Resseguro', 'Saúde/resseguro em grupo específico.'),
 ('22', 'Pessoas EFPC', '2293', 'VIDA EFPC', '2201', 'Vida EFPC', 'Seguro de pessoas vinculado a EFPC.')]

print(f"Rows in embedded mapping: {len(mapping_data)}")


In [0]:
# ============================================================
# 3. CREATE DATAFRAME WITH TARGET STRUCTURE
# ============================================================

base_schema = StructType([
    StructField("cod_grupo_susep", StringType(), False),
    StructField("nome_grupo_susep", StringType(), False),
    StructField("cod_ramo_susep", StringType(), False),
    StructField("nome_ramo_susep", StringType(), False),
    StructField("cod_ramo_tratado", StringType(), False),
    StructField("nome_ramo_tratado", StringType(), False),
    StructField("justificativa_tratamento", StringType(), True),
])

df_base = spark.createDataFrame(mapping_data, schema=base_schema)

df_ref = (
    df_base
    .withColumn("vigencia_inicio", F.to_date(F.lit(vigencia_inicio_default)))
    .withColumn("vigencia_fim", F.lit(vigencia_fim_default).cast(DateType()))
    .withColumn("flag_ativo", F.lit(flag_ativo_default).cast(BooleanType()))
    .withColumn("versao_mapeamento", F.lit(versao_mapeamento_default))
    .withColumn("fonte_regra", F.lit(fonte_regra_default))
    .withColumn("dt_processamento", F.current_timestamp().cast(TimestampType()))
)

hash_columns = [
    "cod_grupo_susep",
    "nome_grupo_susep",
    "cod_ramo_susep",
    "nome_ramo_susep",
    "cod_ramo_tratado",
    "nome_ramo_tratado",
    "vigencia_inicio",
    "vigencia_fim",
    "flag_ativo",
    "versao_mapeamento",
    "justificativa_tratamento",
    "fonte_regra",
]

df_ref = df_ref.withColumn(
    "hash_registro",
    F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in hash_columns]
        ),
        256,
    ),
)

df_ref = df_ref.select(
    "cod_grupo_susep",
    "nome_grupo_susep",
    "cod_ramo_susep",
    "nome_ramo_susep",
    "cod_ramo_tratado",
    "nome_ramo_tratado",
    "vigencia_inicio",
    "vigencia_fim",
    "flag_ativo",
    "versao_mapeamento",
    "justificativa_tratamento",
    "fonte_regra",
    "dt_processamento",
    "hash_registro",
)

print(f"Rows prepared: {df_ref.count()}")
display(df_ref.orderBy("cod_grupo_susep", "cod_ramo_susep"))


In [0]:
# ============================================================
# 4. VALIDATIONS
# ============================================================

expected_rows = len(mapping_data)
actual_rows = df_ref.count()

if actual_rows != expected_rows:
    raise Exception(f"Row count mismatch. Expected {expected_rows}, got {actual_rows}.")

required_columns = [
    "cod_grupo_susep",
    "nome_grupo_susep",
    "cod_ramo_susep",
    "nome_ramo_susep",
    "cod_ramo_tratado",
    "nome_ramo_tratado",
    "vigencia_inicio",
    "flag_ativo",
    "versao_mapeamento",
    "fonte_regra",
    "dt_processamento",
    "hash_registro",
]

for column_name in required_columns:
    null_count = df_ref.filter(F.col(column_name).isNull()).count()
    if null_count > 0:
        raise Exception(f"Column {column_name} has {null_count} null value(s).")

duplicate_ramo = (
    df_ref
    .groupBy("cod_grupo_susep", "cod_ramo_susep")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_ramo.count() > 0:
    display(duplicate_ramo)
    raise Exception("Duplicate SUSEP branch mapping found.")

group_mismatch = df_ref.filter(
    (F.substring(F.col("cod_ramo_susep"), 1, 2) != F.col("cod_grupo_susep")) |
    (F.substring(F.col("cod_ramo_tratado"), 1, 2) != F.col("cod_grupo_susep"))
)

if group_mismatch.count() > 0:
    display(group_mismatch)
    raise Exception("Group mismatch found. A ramo cannot be mapped to a treated ramo from another group.")

print("Validations completed successfully.")


In [0]:
# ============================================================
# 5. WRITE DELTA TABLE
# ============================================================

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")

(
    df_ref.write
    .format("delta")
    .mode(write_mode)
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Table written successfully: {full_table_name}")
print(f"Write mode: {write_mode}")


In [0]:
# ============================================================
# 6. ADD TABLE AND COLUMN COMMENTS
# ============================================================

column_comments = {
    "cod_grupo_susep": "Código do grupo SUSEP",
    "nome_grupo_susep": "Nome do grupo SUSEP",
    "cod_ramo_susep": "Código original do ramo SUSEP",
    "nome_ramo_susep": "Nome original do ramo SUSEP",
    "cod_ramo_tratado": "Código intermediário proposto",
    "nome_ramo_tratado": "Nome padronizado/intermediário",
    "vigencia_inicio": "Data inicial de validade da regra",
    "vigencia_fim": "Data final de validade, se houver",
    "flag_ativo": "Indica se o de/para está ativo",
    "versao_mapeamento": "Versão da tabela de de/para",
    "justificativa_tratamento": "Racional do agrupamento",
    "fonte_regra": "Ex.: SUSEP, análise interna, mercado",
    "dt_processamento": "Data de carga no lake",
    "hash_registro": "Controle de alteração",
}

def sql_string(value):
    return value.replace("'", "''")

try:
    spark.sql(
        f"COMMENT ON TABLE {full_table_name} IS "
        f"'De/para bruto de ramos SUSEP para ramo tratado, usado como referencia de tratamento Bronze -> Silver.'"
    )

    for column_name, column_comment in column_comments.items():
        spark.sql(
            f"ALTER TABLE {full_table_name} ALTER COLUMN {column_name} "
            f"COMMENT '{sql_string(column_comment)}'"
        )

    print("Table and column comments applied successfully.")
except Exception as e:
    print(f"Comments skipped. Table was created, but comments were not applied. Details: {e}")

In [0]:
# ============================================================
# 7. POST LOAD CHECK
# ============================================================

df_loaded = spark.table(full_table_name)

print(f"Loaded rows: {df_loaded.count()}")
print(f"Loaded groups: {df_loaded.select('cod_grupo_susep').distinct().count()}")
print(f"Loaded treated branches: {df_loaded.select('cod_ramo_tratado').distinct().count()}")

display(
    df_loaded
    .groupBy("cod_grupo_susep", "nome_grupo_susep")
    .agg(
        F.countDistinct("cod_ramo_susep").alias("qtd_ramos_susep"),
        F.countDistinct("cod_ramo_tratado").alias("qtd_ramos_tratados"),
    )
    .orderBy("cod_grupo_susep")
)


In [0]:
# ============================================================
# 8. ADD MISSING SUSEP BRANCH CODES
# ============================================================
# This step intentionally does not change the original mapping_data above.
# It only appends missing coramo values identified outside the current de/para.
#
# Source/rationale:
# - Official SUSEP SES BaseCompleta / Ses_ramos.csv for branch names where available.
# - Historical SES compatibility: some coramo values appear without the current 4-digit
#   group+branch representation. In these cases, the raw coramo is mapped to the same
#   treated branch already defined for the equivalent official SUSEP branch code.

versao_mapeamento_missing = "v1.1"
fonte_regra_missing = "SUSEP SES BaseCompleta/Ses_ramos.csv + compatibilizacao historica de coramo"

missing_mapping_data = [('01',
  'Patrimonial',
  '11',
  'INCÊNDIO TRADICIONAL(RUN OFF) [coramo legado 11; ramo oficial 0111]',
  '0101',
  'Incêndio e Compreensivos Patrimoniais',
  'Compatibilização histórica do SES: coramo legado 11 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0111.'),
 ('01',
  'Patrimonial',
  '12',
  'Assistência - Bens em Geral [coramo legado 12; ramo oficial 0112]',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Compatibilização histórica do SES: coramo legado 12 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0112.'),
 ('01',
  'Patrimonial',
  '13',
  'Vidros [coramo legado 13; ramo oficial 0113]',
  '0103',
  'Vidros',
  'Compatibilização histórica do SES: coramo legado 13 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0113.'),
 ('01',
  'Patrimonial',
  '15',
  'ROUBO(RUN OFF) [coramo legado 15; ramo oficial 0115]',
  '0104',
  'Roubo e Tumultos',
  'Compatibilização histórica do SES: coramo legado 15 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0115.'),
 ('01',
  'Patrimonial',
  '17',
  'Tumultos [coramo legado 17; ramo oficial 0117]',
  '0104',
  'Roubo e Tumultos',
  'Compatibilização histórica do SES: coramo legado 17 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0117.'),
 ('06',
  'Transportes',
  '21',
  'Transporte Nacional [coramo legado 21; ramo oficial 0621]',
  '0601',
  'Transporte Nacional e Internacional',
  'Compatibilização histórica do SES: coramo legado 21 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0621.'),
 ('06',
  'Transportes',
  '22',
  'Transporte Internacional [coramo legado 22; ramo oficial 0622]',
  '0601',
  'Transporte Nacional e Internacional',
  'Compatibilização histórica do SES: coramo legado 22 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0622.'),
 ('05',
  'Automóvel',
  '23',
  'RC T.ROD.PAS INTEST OU INT(RUN OFF) [coramo legado 23; ramo oficial 0523]',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'Compatibilização histórica do SES: coramo legado 23 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0523.'),
 ('05',
  'Automóvel',
  '31',
  'Automóvel - Casco [coramo legado 31; ramo oficial 0531]',
  '0504',
  'Automóvel - Casco e Auto Popular',
  'Compatibilização histórica do SES: coramo legado 31 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0531.'),
 ('04',
  'Cascos',
  '33',
  'Marítimos(RUN OFF) [coramo legado 33; ramo oficial 0433]',
  '0401',
  'Cascos Marítimos - Run Off',
  'Compatibilização histórica do SES: coramo legado 33 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0433.'),
 ('02',
  'Riscos Especiais',
  '34',
  'RISCOS DE PETRÓLEO(RUN OFF) [coramo legado 34; ramo oficial 0234]',
  '0201',
  'Riscos de Petróleo - Run Off',
  'Compatibilização histórica do SES: coramo legado 34 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0234.'),
 ('04',
  'Cascos',
  '35',
  'AERONÁUTICOS(RUN OFF) [coramo legado 35; ramo oficial 0435]',
  '0402',
  'Cascos Aeronáuticos',
  'Compatibilização histórica do SES: coramo legado 35 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0435.'),
 ('09',
  'Pessoas Coletivo',
  '36',
  'Perda Certif. Habilit. de Vôo-PCHV [coramo legado 36; ramo oficial 0936]',
  '0902',
  'PCHV',
  'Compatibilização histórica do SES: coramo legado 36 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0936.'),
 ('04',
  'Cascos',
  '37',
  'RESPONSAB CIVIL HANGAR(RUN OFF) [coramo legado 37; ramo oficial 0437]',
  '0403',
  'Responsabilidade Civil Hangar - Run Off',
  'Compatibilização histórica do SES: coramo legado 37 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0437.'),
 ('01',
  'Patrimonial',
  '41',
  'Lucros Cessantes [coramo legado 41; ramo oficial 0141]',
  '0105',
  'Lucros Cessantes',
  'Compatibilização histórica do SES: coramo legado 41 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0141.'),
 ('01',
  'Patrimonial',
  '42',
  'Lucros Cessantes Cobertura Simples [coramo legado 42; ramo oficial 0142]',
  '0105',
  'Lucros Cessantes',
  'Compatibilização histórica do SES: coramo legado 42 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0142.'),
 ('01',
  'Patrimonial',
  '43',
  'Fidelidade [coramo legado 43; ramo oficial 0143]',
  '0106',
  'Fidelidade e Global de Bancos',
  'Compatibilização histórica do SES: coramo legado 43 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0143.'),
 ('05',
  'Automóvel',
  '44',
  'R.C.T.Viag InterPes Tran/ñ(RUN OFF) [coramo legado 44; ramo oficial 0544]',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'Compatibilização histórica do SES: coramo legado 44 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0544.'),
 ('07',
  'Riscos Financeiros',
  '46',
  'Fiança Locatícia [coramo legado 46; ramo oficial 0746]',
  '0704',
  'Fiança Locatícia',
  'Compatibilização histórica do SES: coramo legado 46 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0746.'),
 ('08',
  'Crédito',
  '48',
  'Crédito Interno [coramo legado 48; ramo oficial 0848]',
  '0802',
  'Crédito Interno/Doméstico',
  'Compatibilização histórica do SES: coramo legado 48 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0848.'),
 ('08',
  'Crédito',
  '49',
  'Crédito à Exportação [coramo legado 49; ramo oficial 0849]',
  '0801',
  'Crédito à Exportação',
  'Compatibilização histórica do SES: coramo legado 49 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0849.'),
 ('03',
  'Responsabilidades',
  '51',
  'R. C. Geral [coramo legado 51; ramo oficial 0351]',
  '0304',
  'Responsabilidade Civil Geral',
  'Compatibilização histórica do SES: coramo legado 51 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0351.'),
 ('06',
  'Transportes',
  '52',
  'R. C. Trans. Aéreo Carga - RCTA-C [coramo legado 52; ramo oficial 0652]',
  '0604',
  'RC Transporte de Carga',
  'Compatibilização histórica do SES: coramo legado 52 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0652.'),
 ('05',
  'Automóvel',
  '53',
  'R. C. Facultativa Veículos - RCFV [coramo legado 53; ramo oficial 0553]',
  '0502',
  'Responsabilidade Civil Veicular - Auto',
  'Compatibilização histórica do SES: coramo legado 53 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0553.'),
 ('06',
  'Transportes',
  '54',
  'R.C. Trans. Rodoviário Carga – RCTR-C [coramo legado 54; ramo oficial 0654]',
  '0604',
  'RC Transporte de Carga',
  'Compatibilização histórica do SES: coramo legado 54 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0654.'),
 ('06',
  'Transportes',
  '55',
  'RC Trans Desaparec Carga RC-DC [coramo legado 55; ramo oficial 0655]',
  '0604',
  'RC Transporte de Carga',
  'Compatibilização histórica do SES: coramo legado 55 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0655.'),
 ('06',
  'Transportes',
  '56',
  'R.C. Trans. Aquaviário Carga – RCA-C [coramo legado 56; ramo oficial 0656]',
  '0604',
  'RC Transporte de Carga',
  'Compatibilização histórica do SES: coramo legado 56 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0656.'),
 ('04',
  'Cascos',
  '57',
  'D. P. E. M.(RUN OFF) [coramo legado 57; ramo oficial 0457]',
  '0404',
  'DPEM - Run Off',
  'Compatibilização histórica do SES: coramo legado 57 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0457.'),
 ('11',
  'Rural',
  '61',
  'Agrícola [coramo legado 61; ramo oficial 1161]',
  '1101',
  'Agrícola',
  'Compatibilização histórica do SES: coramo legado 61 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1161.'),
 ('11',
  'Rural',
  '62',
  'Penhor Rural [coramo legado 62; ramo oficial 1162]',
  '1106',
  'Bens, Benfeitorias e Penhor Rural',
  'Compatibilização histórica do SES: coramo legado 62 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1162.'),
 ('11',
  'Rural',
  '63',
  'Penhor Rural Inst. Fin Pub(RUN OFF) [coramo legado 63; ramo oficial 1163]',
  '1106',
  'Bens, Benfeitorias e Penhor Rural',
  'Compatibilização histórica do SES: coramo legado 63 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1163.'),
 ('11',
  'Rural',
  '64',
  'Seguros Animais [coramo legado 64; ramo oficial 1164]',
  '1102',
  'Pecuário e Animais',
  'Compatibilização histórica do SES: coramo legado 64 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1164.'),
 ('11',
  'Rural',
  '65',
  'Compreensivo de Florestas [coramo legado 65; ramo oficial 1165]',
  '1104',
  'Florestas',
  'Compatibilização histórica do SES: coramo legado 65 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1165.'),
 ('10',
  'Habitacional',
  '66',
  'Seg.Hab.Sist.Fin. da Habit(RUN OFF) [coramo legado 66; ramo oficial 1066]',
  '1003',
  'Habitacional - Run Off/SFH',
  'Compatibilização histórica do SES: coramo legado 66 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1066.'),
 ('01',
  'Patrimonial',
  '67',
  'Riscos de Engenharia [coramo legado 67; ramo oficial 0167]',
  '0107',
  'Riscos de Engenharia, Nomeados e Operacionais',
  'Compatibilização histórica do SES: coramo legado 67 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0167.'),
 ('10',
  'Habitacional',
  '68',
  'HABITACIONAL - FORA DO SFH(RUN OFF) [coramo legado 68; ramo oficial 1068]',
  '1003',
  'Habitacional - Run Off/SFH',
  'Compatibilização histórica do SES: coramo legado 68 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1068.'),
 ('09',
  'Pessoas Coletivo',
  '69',
  'Viagem [coramo legado 69; ramo oficial 0969]',
  '0903',
  'Viagem',
  'Compatibilização histórica do SES: coramo legado 69 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0969.'),
 ('01',
  'Patrimonial',
  '71',
  'Riscos Diversos [coramo legado 71; ramo oficial 0171]',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Compatibilização histórica do SES: coramo legado 71 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0171.'),
 ('02',
  'Riscos Especiais',
  '72',
  'RISCOS NUCLEARES(RUN OFF) [coramo legado 72; ramo oficial 0272]',
  '0202',
  'Riscos Nucleares - Run Off',
  'Compatibilização histórica do SES: coramo legado 72 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0272.'),
 ('01',
  'Patrimonial',
  '73',
  'Global de Bancos [coramo legado 73; ramo oficial 0173]',
  '0106',
  'Fidelidade e Global de Bancos',
  'Compatibilização histórica do SES: coramo legado 73 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0173.'),
 ('02',
  'Riscos Especiais',
  '74',
  'SATÉLITES(RUN OFF) [coramo legado 74; ramo oficial 0274]',
  '0203',
  'Satélites - Run Off',
  'Compatibilização histórica do SES: coramo legado 74 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0274.'),
 ('07',
  'Riscos Financeiros',
  '75',
  'Garantia Segurado - Setor Público [coramo legado 75; ramo oficial 0775]',
  '0702',
  'Garantia',
  'Compatibilização histórica do SES: coramo legado 75 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0775.'),
 ('01',
  'Patrimonial',
  '76',
  'Riscos Diversos - Planos Conjugados [coramo legado 76; ramo oficial 0176]',
  '0102',
  'Riscos Diversos e Assistência a Bens',
  'Compatibilização histórica do SES: coramo legado 76 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0176.'),
 ('12',
  'Outros',
  '79',
  'Seguros no Exterior(RUN OFF) [coramo legado 79; ramo oficial 1279]',
  '1201',
  'Exterior e Sucursais no Exterior',
  'Compatibilização histórica do SES: coramo legado 79 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1279.'),
 ('09',
  'Pessoas Coletivo',
  '80',
  'Educacional [coramo legado 80; ramo oficial 0980]',
  '0905',
  'Educacional',
  'Compatibilização histórica do SES: coramo legado 80 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0980.'),
 ('09',
  'Pessoas Coletivo',
  '81',
  'ACIDENTES PESS - INDIVID(RUN OFF) [coramo legado 81; ramo oficial 0981]',
  '0906',
  'Acidentes Pessoais',
  'Compatibilização histórica do SES: coramo legado 81 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0981.'),
 ('09',
  'Pessoas Coletivo',
  '82',
  'Acidentes Pessoais [coramo legado 82; ramo oficial 0982]',
  '0906',
  'Acidentes Pessoais',
  'Compatibilização histórica do SES: coramo legado 82 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0982.'),
 ('09',
  'Pessoas Coletivo',
  '83',
  'Dotal Misto [coramo legado 83; ramo oficial 0983]',
  '0907',
  'Dotais',
  'Compatibilização histórica do SES: coramo legado 83 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0983.'),
 ('09',
  'Pessoas Coletivo',
  '84',
  'Doenças Graves ou Doença Terminal [coramo legado 84; ramo oficial 0984]',
  '0908',
  'Doenças Graves ou Doença Terminal',
  'Compatibilização histórica do SES: coramo legado 84 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0984.'),
 ('12',
  'Outros',
  '85',
  'Saúde - Ressegurador Loc (RUN OFF) [coramo legado 85; ramo oficial 1285]',
  '1202',
  'Saúde',
  'Compatibilização histórica do SES: coramo legado 85 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1285.'),
 ('12',
  'Outros',
  '86',
  'Saúde Individual [coramo legado 86; ramo oficial 1286]',
  '1202',
  'Saúde',
  'Compatibilização histórica do SES: coramo legado 86 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1286.'),
 ('12',
  'Outros',
  '87',
  'Saúde Grupal [coramo legado 87; ramo oficial 1287]',
  '1202',
  'Saúde',
  'Compatibilização histórica do SES: coramo legado 87 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1287.'),
 ('05',
  'Automóvel',
  '88',
  'DPVAT [coramo legado 88; ramo oficial 0588]',
  '0506',
  'DPVAT',
  'Compatibilização histórica do SES: coramo legado 88 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0588.'),
 ('05',
  'Automóvel',
  '89',
  'DPVAT(RUN OFF) [coramo legado 89; ramo oficial 0589]',
  '0506',
  'DPVAT',
  'Compatibilização histórica do SES: coramo legado 89 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0589.'),
 ('09',
  'Pessoas Coletivo',
  '90',
  'Eventos Aleatórios [coramo legado 90; ramo oficial 0990]',
  '0910',
  'Eventos Aleatórios',
  'Compatibilização histórica do SES: coramo legado 90 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0990.'),
 ('09',
  'Pessoas Coletivo',
  '91',
  'Vida(RUN OFF) [coramo legado 91; ramo oficial 0991]',
  '0911',
  'Vida e VG/APC',
  'Compatibilização histórica do SES: coramo legado 91 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0991.'),
 ('09',
  'Pessoas Coletivo',
  '93',
  'Vida em Grupo [coramo legado 93; ramo oficial 0993]',
  '0911',
  'Vida e VG/APC',
  'Compatibilização histórica do SES: coramo legado 93 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0993.'),
 ('09',
  'Pessoas Coletivo',
  '97',
  'VG/APC [coramo legado 97; ramo oficial 0997]',
  '0911',
  'Vida e VG/APC',
  'Compatibilização histórica do SES: coramo legado 97 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 0997.'),
 ('12',
  'Outros',
  '99',
  'SUCURSAIS NO EXTERIOR(RUN OFF) [coramo legado 99; ramo oficial 1299]',
  '1201',
  'Exterior e Sucursais no Exterior',
  'Compatibilização histórica do SES: coramo legado 99 tratado pelo mesmo agrupamento do ramo '
  'SUSEP oficial 1299.'),
 ('09',
  'Pessoas Coletivo',
  '996',
  'Seguro de Vida Universal [coramo sem zero; ramo oficial 0996]',
  '0996',
  'Seguro de Vida Universal',
  'Compatibilização do SES: coramo 996 tratado pelo mesmo agrupamento do ramo SUSEP oficial 0996.'),
 ('13',
  'Pessoas Individual',
  '1396',
  'Seguro de Vida Universal',
  '1396',
  'Seguro de Vida Universal',
  'Ramo SUSEP do grupo Pessoas Individual; mantido como ramo tratado próprio.'),
 ('22',
  'Pessoas EFPC',
  '2201',
  'Sobrevivência de Assistido',
  '2201',
  'Sobrevivência de Assistido',
  'Ramo SUSEP do grupo Pessoas EFPC; mantido como ramo tratado próprio.'),
 ('22',
  'Pessoas EFPC',
  '2202',
  'Fluxo Biométrico',
  '2202',
  'Fluxo Biométrico',
  'Ramo SUSEP do grupo Pessoas EFPC; mantido como ramo tratado próprio.'),
 ('22',
  'Pessoas EFPC',
  '2203',
  'Índice Biométrico',
  '2203',
  'Índice Biométrico',
  'Ramo SUSEP do grupo Pessoas EFPC; mantido como ramo tratado próprio.')]

df_missing_base = spark.createDataFrame(missing_mapping_data, schema=base_schema)

df_missing = (
    df_missing_base
    .withColumn("vigencia_inicio", F.to_date(F.lit(vigencia_inicio_default)))
    .withColumn("vigencia_fim", F.lit(vigencia_fim_default).cast(DateType()))
    .withColumn("flag_ativo", F.lit(flag_ativo_default).cast(BooleanType()))
    .withColumn("versao_mapeamento", F.lit(versao_mapeamento_missing))
    .withColumn("fonte_regra", F.lit(fonte_regra_missing))
    .withColumn("dt_processamento", F.current_timestamp().cast(TimestampType()))
)

df_missing = df_missing.withColumn(
    "hash_registro",
    F.sha2(
        F.concat_ws(
            "||",
            *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in hash_columns]
        ),
        256,
    ),
)

df_missing = df_missing.select(
    "cod_grupo_susep",
    "nome_grupo_susep",
    "cod_ramo_susep",
    "nome_ramo_susep",
    "cod_ramo_tratado",
    "nome_ramo_tratado",
    "vigencia_inicio",
    "vigencia_fim",
    "flag_ativo",
    "versao_mapeamento",
    "justificativa_tratamento",
    "fonte_regra",
    "dt_processamento",
    "hash_registro",
)

expected_missing_rows = len(missing_mapping_data)
actual_missing_rows = df_missing.count()

if actual_missing_rows != expected_missing_rows:
    raise Exception(f"Missing branch row count mismatch. Expected {expected_missing_rows}, got {actual_missing_rows}.")

duplicate_missing_ramo = (
    df_missing
    .groupBy("cod_ramo_susep")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_missing_ramo.count() > 0:
    display(duplicate_missing_ramo)
    raise Exception("Duplicate cod_ramo_susep found in missing_mapping_data.")

df_existing_ramos = spark.table(full_table_name).select("cod_ramo_susep").distinct()

df_missing_to_insert = (
    df_missing
    .join(df_existing_ramos, on=["cod_ramo_susep"], how="left_anti")
    .select(df_missing.columns)
)

rows_to_insert = df_missing_to_insert.count()

if rows_to_insert > 0:
    (
        df_missing_to_insert.write
        .format("delta")
        .mode("append")
        .saveAsTable(full_table_name)
    )

print(f"Missing branch rows prepared: {actual_missing_rows}")
print(f"Missing branch rows inserted: {rows_to_insert}")
print(f"Final loaded rows: {spark.table(full_table_name).count()}")

display(
    spark.table(full_table_name)
    .filter(F.col("cod_ramo_susep").isin([row[2] for row in missing_mapping_data]))
    .orderBy("cod_ramo_susep")
)
